In [ ]:
# ------------------------------------
#  Mount Google Drive
# ------------------------------------

# Import library to access Google Drive in Colab
from google.colab import drive

# Mount Google Drive to access files stored in your drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================
# INSTALL LIBRARIES
# ================================
!pip install flask pyngrok transformers torch flask-cors
# Paste your own token here ↓↓↓
!ngrok config add-authtoken 3BOLwskkCur8xzBIsmhzm8gYUnt_2ttoqGTLHqERNxXstF2LZ

# ================================
# IMPORTS
# ================================
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import re
import os

# ================================
# INIT APP
# ================================
app = Flask(__name__)
CORS(app)

# ================================
# LOAD MODEL
# ================================
model_path = "/content/drive/MyDrive/Fake News Detection /Processed Data/Model Training"

print("Files in model folder:", os.listdir(model_path))

# 🔥 Use SAME tokenizer as training
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# ================================
# TEXT CLEANING (IMPORTANT)
# ================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ================================
# PREDICTION FUNCTION
# ================================
def predict_news(text):

    text = clean_text(text)   # 🔥 MUST MATCH TRAINING

    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 🔥 Use probabilities (better than argmax)
    probs = torch.softmax(logits, dim=1)

    fake_prob = probs[0][0].item()
    real_prob = probs[0][1].item()

    print("Fake:", fake_prob, "Real:", real_prob)

    # 🔥 Final decision
    if real_prob > fake_prob:
        return "REAL NEWS ✅"
    else:
        return "FAKE NEWS ❌"

# ================================
# API ROUTE
# ================================
@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    text = data["text"]

    result = predict_news(text)
    return jsonify({"result": result})

# ================================
# START NGROK
# ================================
public_url = ngrok.connect(5000)
print("🌐 COPY THIS URL:", public_url)

# ================================
# RUN APP
# ================================
app.run(port=5000)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Files in model folder: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

🌐 COPY THIS URL: NgrokTunnel: "https://unannihilable-aponeurotic-kailani.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Mar/2026 12:57:54] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Mar/2026 12:57:55] "POST /predict HTTP/1.1" 200 -


Fake: 0.9701939225196838 Real: 0.02980613149702549


INFO:werkzeug:127.0.0.1 - - [25/Mar/2026 12:58:29] "OPTIONS /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Mar/2026 12:58:30] "POST /predict HTTP/1.1" 200 -


Fake: 0.09163305908441544 Real: 0.9083669781684875
